# N-01 — Estados, Operadores e Medições

- **Trilha:** Núcleo da QC
- **Milestone:** 1 — Linguagem mínima

<div style="border-left: 2px solid #6b7280; padding: 0.5em 1em 0.5em 1.1em; margin: 0.8em 0 1.2em 0; background: rgba(107, 114, 128, 0.05); border-radius: 0 3px 3px 0;">
  <span style="font-size: 0.65em; font-weight: 700; letter-spacing: 0.13em; text-transform: uppercase; color: #9ca3af;">ETIMOLOGIA </span><br>

  <span style="font-size: 0.88em; line-height: 1.65;"> <b>Estado</b> vem do latim <i>status</i>, de <i>stare</i> (“estar de pé”, “permanecer”). A ideia original é a de uma condição que se mantém. <b>Em termos gerais</b>, “estado” descreve como algo está em um dado momento — a configuração atual de um sistema, independentemente de como ele chegou ali.</span><br>
  
  <span style="font-size: 0.88em; line-height: 1.65;"> <b>Operador</b> vem do latim <i>operari</i> (“trabalhar”, “agir”), de <i>opus</i> (“obra”, “trabalho”). A ideia original é a de executar uma ação. <b>Em termos gerais</b>, “operador” é algo que age sobre um objeto e o transforma segundo regras definidas.</span><br>
  
  <span style="font-size: 0.88em; line-height: 1.65;"><b>Medição</b> vem do latim <i>mensura</i>, de <i>metiri</i> (“medir”, “avaliar por comparação”). A ideia original é a de comparar com um padrão. <b>Em termos gerais</b>, “medição” é o processo de extrair uma informação observável de algo, traduzindo uma característica em um resultado comparável.</span>
</div>

## Afirmação introdutória

O `estado` de um qubit é um vetor de norma 1 num espaço complexo de dimensão 2. `Operar` sobre ele é multiplicar esse vetor por uma matriz que preserva a norma — um `unitário`. `Medir` é projetar o vetor numa base: a probabilidade de cada resultado é o módulo ao quadrado da projeção, e após a medição o estado colapsa para o vetor da base correspondente.

Em N-00 vimos tudo isso acontecer com a moeda e as ondas. Neste notebook, damos nome e estrutura a cada peça — de modo que você possa descrever qualquer experimento quântico sem ambiguidade.

## 1. O Estado como Vetor

Em N-00, o estado do qubit era $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ com $|\alpha|^2 + |\beta|^2 = 1$. Isso não é uma conveniência de notação — é um vetor num espaço vetorial complexo de dimensão 2.

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \quad \alpha, \beta \in \mathbb{C}, \quad |\alpha|^2 + |\beta|^2 = 1$$

<div style="margin: -0.6em 0 1.4em 0; font-size: 0.87em; color: #6b7280; line-height: 1.65; padding-left: 1.1em; border-left: 2px solid #374151;">
  Os vetores da base — |0⟩ e |1⟩ — são os únicos resultados possíveis de uma medição. O estado antes de medir é uma combinação linear deles; as amplitudes α e β dizem quanto de cada resultado está "embutido" no estado.
</div>

A restrição $|\alpha|^2 + |\beta|^2 = 1$ não é física — é geométrica: ela diz que o vetor estado tem norma 1, o que garante que as probabilidades somem 100%.

A estrutura desse espaço — produto interno, norma, completude — é o que chamamos de **espaço de Hilbert**. Para um qubit, ele é simplesmente $\mathbb{C}^2$. Para $n$ qubits, $\mathbb{C}^{2^n}$.

In [1]:
import numpy as np

# ── Alguns estados canônicos ─────────────────────────────────────────────────
ket_0 = np.array([1, 0], dtype=complex)   # |0⟩
ket_1 = np.array([0, 1], dtype=complex)   # |1⟩
ket_p = np.array([1, 1], dtype=complex) / np.sqrt(2)   # |+⟩ = H|0⟩
ket_m = np.array([1, -1], dtype=complex) / np.sqrt(2)  # |−⟩ = H|1⟩

def norma(psi):
    return np.sqrt(np.conj(psi) @ psi).real

for nome, psi in [('|0⟩', ket_0), ('|1⟩', ket_1), ('|+⟩', ket_p), ('|−⟩', ket_m)]:
    print(f'{nome}  →  vetor {psi}   norma = {norma(psi):.4f}')

# ── Produto interno ──────────────────────────────────────────────────────────
print()
print('⟨0|1⟩ =', np.conj(ket_0) @ ket_1)   # ortogonais → 0
print('⟨+|−⟩ =', np.conj(ket_p) @ ket_m)   # ortogonais → 0
print('⟨0|+⟩ =', np.conj(ket_0) @ ket_p)   # |⟨0|+⟩|² = 0.5 → P(|0⟩) ao medir |+⟩

|0⟩  →  vetor [1.+0.j 0.+0.j]   norma = 1.0000
|1⟩  →  vetor [0.+0.j 1.+0.j]   norma = 1.0000
|+⟩  →  vetor [0.70710678+0.j 0.70710678+0.j]   norma = 1.0000
|−⟩  →  vetor [ 0.70710678+0.j -0.70710678+0.j]   norma = 1.0000

⟨0|1⟩ = 0j
⟨+|−⟩ = 0j
⟨0|+⟩ = (0.7071067811865475+0j)


<a href="../matematica/M-00.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ espaços vetoriais, produto interno e espaço de Hilbert</a>

## 2. Operadores como Matrizes

Um operador em $\mathbb{C}^2$ é uma transformação linear — na prática, uma matriz $2 \times 2$. Dois tipos aparecem o tempo todo:

**Unitários** — preservam a norma do estado:
$$U^\dagger U = I \quad \Longleftrightarrow \quad \|U|\psi\rangle\| = \||\psi\rangle\|$$

<div style="margin: -0.6em 0 1.4em 0; font-size: 0.87em; color: #6b7280; line-height: 1.65; padding-left: 1.1em; border-left: 2px solid #374151;">
  Preservar norma = preservar a soma das probabilidades = a operação é reversível. Em N-00 vimos que inversão de fase e recombinação conservavam o total de probabilidade — porque ambas são unitárias.
</div>

**Hermitianos** — autovalores reais; representam observáveis (o que pode ser medido):
$$H^\dagger = H \quad \Longrightarrow \quad \text{autovalores} \in \mathbb{R}$$

As matrizes de Pauli são os tijolos básicos:

$$X = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix}, \quad Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

Cada uma é Hermitiana (observável) *e* unitária (operação). O gate de Hadamard:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$$

é o que usamos em N-00 para colocar em superposição e para recombinar — sem saber o nome.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Matrizes de Pauli e Hadamard ─────────────────────────────────────────────
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

ket_0 = np.array([1, 0], dtype=complex)

# ── Verificar unitaridade: U†U = I ───────────────────────────────────────────
for nome, U in [('X', X), ('Y', Y), ('Z', Z), ('H', H)]:
    eh_unitario = np.allclose(U.conj().T @ U, np.eye(2))
    print(f'{nome}: unitário = {eh_unitario}')

# ── H coloca |0⟩ em superposição ────────────────────────────────────────────
print()
print('H|0⟩ =', H @ ket_0)
print('H(H|0⟩) =', H @ (H @ ket_0), '  (volta para |0⟩ — H é sua própria inversa)')

# ── Prática: norma é preservada ──────────────────────────────────────────────
psi = np.array([0.6 + 0.2j, 0.8 - 0.1j])
psi /= np.linalg.norm(psi)   # normalizar

for nome, U in [('X', X), ('Y', Y), ('Z', Z), ('H', H)]:
    norma_antes = np.linalg.norm(psi)
    norma_depois = np.linalg.norm(U @ psi)
    print(f'{nome}|ψ⟩: norma antes = {norma_antes:.4f}, depois = {norma_depois:.4f}')

## 3. Medição — a Regra de Born

Medir um estado $|\psi\rangle$ numa base $\{|b_0\rangle, |b_1\rangle\}$ é perguntar: em qual dos vetores da base o estado vai "colapsar"?

A **regra de Born** diz a probabilidade de cada resposta:

$$P(b_k) = |\langle b_k | \psi \rangle|^2$$

<div style="margin: -0.6em 0 1.4em 0; font-size: 0.87em; color: #6b7280; line-height: 1.65; padding-left: 1.1em; border-left: 2px solid #374151;">
  ⟨b_k|ψ⟩ é o produto interno entre o vetor da base e o estado — geometricamente, a "projeção" de |ψ⟩ na direção de |b_k⟩. O módulo ao quadrado dessa projeção é a probabilidade. Por isso medir não é neutro: projeta o estado em algo menor.
</div>

Após a medição, o estado colapsa: se o resultado foi $b_k$, o estado agora é $|b_k\rangle$. A superposição foi destruída.

<div style="border-left: 2px solid #f59e0b; padding: 0.5em 1em 0.5em 1.1em; margin: 0.8em 0 1.2em 0; background: rgba(245, 158, 11, 0.05); border-radius: 0 3px 3px 0;">
  <span style="font-size: 0.65em; font-weight: 700; letter-spacing: 0.13em; text-transform: uppercase; color: #f59e0b;">atenção</span><br>
  <span style="font-size: 0.88em; line-height: 1.65;">A base em que você mede importa. O mesmo estado |+⟩ dá resultados uniformes se medido na base Z (|0⟩, |1⟩), mas é determinístico se medido na base X (|+⟩, |−⟩). Medir é sempre uma pergunta que depende de quem pergunta.</span>
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
ket_p = np.array([1, 1], dtype=complex) / np.sqrt(2)   # |+⟩
ket_m = np.array([1, -1], dtype=complex) / np.sqrt(2)  # |−⟩

psi = ket_p   # estado a medir: |+⟩

# ── Medir na base Z ──────────────────────────────────────────────────────────
P0_z = abs(np.conj(ket_0) @ psi)**2   # P(|0⟩)
P1_z = abs(np.conj(ket_1) @ psi)**2   # P(|1⟩)

# ── Medir na base X ──────────────────────────────────────────────────────────
P0_x = abs(np.conj(ket_p) @ psi)**2   # P(|+⟩)
P1_x = abs(np.conj(ket_m) @ psi)**2   # P(|−⟩)

print('Estado: |+⟩')
print()
print('Medição na base Z (computacional):')
print(f'  P(|0⟩) = {P0_z:.2f}   P(|1⟩) = {P1_z:.2f}')
print()
print('Medição na base X (de Hadamard):')
print(f'  P(|+⟩) = {P0_x:.2f}   P(|−⟩) = {P1_x:.2f}')
print()
print('Mesma conclusão: a base em que você mede determina o resultado.')

# ── Prática visual: amplitudes vs. probabilidades para |+⟩ nas duas bases ───
fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='#111')
fig.patch.set_facecolor('#111')

for ax in axes:
    ax.set_facecolor('#111')
    ax.tick_params(colors='#9ca3af')
    for spine in ax.spines.values():
        spine.set_edgecolor('#374151')
    ax.set_ylim(0, 1.15)

# Base Z
axes[0].bar(['|0⟩', '|1⟩'], [P0_z, P1_z], color=['steelblue', 'tomato'], alpha=0.85)
axes[0].set_title('|+⟩ medido na base Z', color='white', pad=10)
axes[0].set_ylabel('probabilidade', color='#9ca3af')
for i, v in enumerate([P0_z, P1_z]):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', color='white', fontsize=11)

# Base X
axes[1].bar(['|+⟩', '|−⟩'], [P0_x, P1_x], color=['steelblue', 'tomato'], alpha=0.85)
axes[1].set_title('|+⟩ medido na base X', color='white', pad=10)
for i, v in enumerate([P0_x, P1_x]):
    axes[1].text(i, v + 0.02, f'{v:.0%}', ha='center', color='white', fontsize=11)

plt.suptitle('O mesmo estado — bases de medição diferentes → resultados diferentes',
             color='#9ca3af', fontsize=10)
plt.tight_layout()
plt.show()

<a href="../digressoes/D-01.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ regra de Born, colapso e por que a medição é irreversível</a>

## 4. Por que as Amplitudes Precisam ser Complexas

Em N-00, as amplitudes eram reais — e isso bastou para mostrar interferência. Mas o conjunto de interferências possíveis com amplitudes reais é estritamente menor do que com amplitudes complexas.

Um número complexo $\alpha = r e^{i\theta}$ carrega um módulo $r$ (que dita a probabilidade) e uma fase $\theta$ (que determina como a amplitude interfere com outras). Com amplitudes reais, a fase só pode ser $0$ ou $\pi$ — "concordar" ou "discordar". Com amplitudes complexas, a fase pode ser qualquer ângulo no círculo unitário.

Consequência prática: certas portas quânticas — necessárias para computação universal — só existem com fases complexas. A porta $T = \text{diag}(1, e^{i\pi/4})$, por exemplo, introduz uma fase de $\pi/4$ que não tem equivalente real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── T gate: fase complexa que não tem equivalente real ───────────────────────
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

ket_p = np.array([1, 1], dtype=complex) / np.sqrt(2)
T_psi = T @ ket_p

print('T|+⟩ =', T_psi)
print('Módulos:', np.abs(T_psi))
print('Fases (graus):', np.angle(T_psi, deg=True))
print('Probabilidades iguais:', np.abs(T_psi)**2)
print()
print('A T gate não muda as probabilidades — só a fase relativa.')
print('Mas essa fase torna certas interferências impossíveis com amplitudes reais.')

# ── Visualização: fase relativa no plano complexo ────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5), facecolor='#111')
ax.set_facecolor('#111')
ax.tick_params(colors='#9ca3af')
for spine in ax.spines.values():
    spine.set_edgecolor('#374151')

theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), color='#374151', lw=1)
ax.axhline(0, color='#374151', lw=0.6)
ax.axvline(0, color='#374151', lw=0.6)

for amp, label, color in [(ket_p[1], '|+⟩ (antes)', 'steelblue'),
                           (T_psi[1], 'T|+⟩ (depois)', 'tomato')]:
    ax.annotate('', xy=(amp.real, amp.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(amp.real + 0.05, amp.imag + 0.05, label,
            color=color, fontsize=9)

ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_title('Fase da amplitude de |1⟩ antes e depois de T', color='white', pad=10)
plt.tight_layout()
plt.show()

<a href="../digressoes/D-02.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ produtos tensoriais — como estados de múltiplos qubits se combinam</a>

## 5. O que este notebook respondeu

Você agora tem a linguagem mínima para descrever qualquer experimento quântico sem ambiguidade:

- Um **estado** é um vetor de norma 1 em $\mathbb{C}^{2^n}$. Para um qubit, é um ponto na esfera de Bloch.
- Uma **operação** é uma matriz unitária: preserva a norma, é reversível, mapeia estados válidos em estados válidos.
- Uma **medição** projeta o estado numa base. A probabilidade de cada resultado é $|\langle b_k|\psi\rangle|^2$ — a regra de Born. Após medir, a superposição acabou.
- As amplitudes precisam ser **complexas** para que o espaço de interferências possíveis seja rico o suficiente para computação universal.

Em N-00, a moeda, as ondas e a interferência eram analogias. Agora você tem o formalismo que torna tudo aquilo preciso.

## 6. O que este notebook não respondeu

Próximo capítulo do núcleo:

- O que o **espectro** de um operador unitário diz sobre a dinâmica que ele gera.
- Por que os autovalores de um unitário estão na circunferência ($|\lambda| = 1$) e o que isso significa para a evolução temporal.
- Como um circuito consegue **extrair informação espectral** de um operador sem calculá-la diretamente.

Matemática:

- O que é um espaço de Hilbert com rigor — <a href="../matematica/M-00.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ M-00</a>
- Teoria espectral: autovalores, autovetores, decomposição espectral — <a href="../matematica/M-01.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ M-01</a>

Digressão:

- Regra de Born e colapso com mais profundidade — <a href="../digressoes/D-01.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ D-01</a>
- Produtos tensoriais para múltiplos qubits — <a href="../digressoes/D-02.ipynb" style="font-size: 0.82em; color: #6b7280; text-decoration: none; border-bottom: 1px dashed #6b7280; white-space: nowrap;">↳ D-02</a>